In [15]:
import numpy as np
import pandas as pd
from trading_rookie.data.get_data import DataDownloader

dd = DataDownloader()
spy_px = dd.yahoo("SPY", "2016-01-01", "2025-12-31")["Adj_close"]

close_df = pd.read_csv("outputs/close_df.csv", index_col=0)
universe50 = pd.read_csv("outputs/universe50.csv")
# close_df[["NVDA", "AAPL"]].head()

Successfully got stock data for SPY


In [ ]:
def compute_panel_features(close_df: pd.DataFrame, spy_close: pd.Series):
    # align dates
    close = close_df.sort_index()
    spy = spy_close.sort_index()
    close, spy = close.align(spy, join="inner", axis=0)

    log_ret = np.log(close).diff()              # (T x N)
    spy_ret = np.log(spy).diff()                # (T,)

    mom63 = log_ret.rolling(63).sum()           # (T x N)
    vol20 = log_ret.rolling(20).std() * np.sqrt(252)

    # beta60 vs SPY: rolling cov(stock, spy) / rolling var(spy)
    # pandas can do rolling cov with Series vs each column via apply
    spy_var60 = spy_ret.rolling(60).var()

    # rolling covariance for each column with spy_ret
    cov60 = log_ret.rolling(60).cov(spy_ret)    # returns (T x N) if spy_ret is Series
    beta60 = cov60.div(spy_var60, axis=0)

    return {
        "close": close,
        "log_ret": log_ret,
        "mom63": mom63,
        "vol20": vol20,
        "beta60": beta60,
        "spy_ret": spy_ret,
    }

def build_target_weights_panel(
    t: pd.Timestamp,
    mom63: pd.DataFrame,
    vol20: pd.DataFrame,
    beta60: pd.DataFrame,
    returns_panel: pd.DataFrame,     # log_ret panel, includes SPY column? 這裡先用股票+SPY的returns
    beta_target=0.3,
    vol_target=0.12,
    pos_cap=0.20,
    gross_cap=1.50,
    top_n=10,
    cov_lookback=120,                # 用來估cov的視窗
):
    # 1) Selection
    x = mom63.loc[t].dropna()
    if x.empty:
        return pd.Series(dtype=float)

    top = x.sort_values(ascending=False).head(top_n).index.tolist()

    # 2) inv-vol weights on long leg
    v = vol20.loc[t, top].replace([np.inf, -np.inf], np.nan).dropna()
    if v.empty:
        return pd.Series(dtype=float)

    inv = 1.0 / v
    w_long = inv / inv.sum()  # long weights sum=1

    # 3) beta hedge with SPY (SPY beta≈1)
    b = beta60.loc[t, w_long.index].fillna(0.0)
    beta_p = float((w_long * b).sum())
    w_spy = beta_target - beta_p

    w = w_long.copy()
    w["SPY"] = w_spy

    # 4) vol targeting (optional but you want it)
    # build covariance using last cov_lookback daily returns up to t
    cols = list(w.index)
    r = returns_panel[cols].loc[:t].tail(cov_lookback).dropna()
    if len(r) >= 30:
        cov = r.cov().values  # daily cov
        w_vec = w.values.reshape(-1, 1)
        var = float(w_vec.T @ cov @ w_vec)
        port_vol = np.sqrt(max(var, 0.0)) * np.sqrt(252)
        if np.isfinite(port_vol) and port_vol > 0:
            k = vol_target / port_vol
            w = w * k

    # 5) constraints
    w = w.clip(lower=-pos_cap, upper=pos_cap)
    gross = float(np.abs(w).sum())
    if gross > gross_cap and gross > 0:
        w *= (gross_cap / gross)

    return w.sort_index()

In [ ]:
feat = compute_panel_features(close_df, spy_px)

log_ret = feat["log_ret"]
spy_ret = feat["spy_ret"]

returns_panel = log_ret.copy()
returns_panel["SPY"] = spy_ret 